### Restart and Run All Cells

In [2]:
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

year = 2026
quarter = 1
current_time = datetime.now()
formatted_time = current_time.strftime("%Y-%m-%d %H:%M:%S")
print(formatted_time )

2026-05-11 22:23:44


In [3]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',   
                'latest_amt':'{:,}','previous_amt':'{:,}','inc_amt':'{:,}',
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%','inc_pct':'{:.2f}%'
              }

In [4]:
sql = """
SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = %s AND quarter <= %s) 
OR (year = %s-1 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC
"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = 2026 AND quarter <= 1) 
OR (year = 2026-1 AND quarter >= 1+1)
ORDER BY year DESC, quarter DESC



In [5]:
dfc = pd.read_sql(sql, conlt)
dfc["Counter"] = 1
dfc_grp = dfc.groupby(["name"], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp["Counter"] == 4]
dfc_grp.shape

(54, 5)

In [6]:
sql = """
SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = %s AND quarter <= %s-1) 
OR (year = %s-1 AND quarter >= %s) 
ORDER BY year DESC, quarter DESC"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = 2026 AND quarter <= 1-1) 
OR (year = 2026-1 AND quarter >= 1) 
ORDER BY year DESC, quarter DESC


In [7]:
dfp = pd.read_sql(sql, conlt)
dfp["Counter"] = 1
dfp_grp = dfp.groupby(["name"], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp["Counter"] == 4]
dfp_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8100,10,"10,827,771",4
1,ACE,8100,10,"798,614",4
2,ADVANC,8100,10,"47,885,902",4
4,AH,8100,10,"731,428",4
5,AIE,8100,10,"21,904",4


In [8]:
dfp.name.unique().shape

(219,)

In [9]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
df_percent.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q1,"6,831,513","10,827,771","-3,996,258",-36.91%
1,ADVANC,2026,Q1,"50,797,881","47,885,902","2,911,979",6.08%
2,AIT,2026,Q1,"581,809","581,113",696,0.12%
3,AOT,2026,Q1,"17,433,533","18,125,205","-691,672",-3.82%
4,ASIAN,2026,Q1,"604,361","681,832","-77,471",-11.36%


In [10]:
# Create the SQL query with parameter binding
sql = text("DELETE FROM qt_profits WHERE year = :year AND quarter = :quarter")

# Execute the query with parameters
params = {'year': year, 'quarter': f'Q{quarter}'}
rp = conlt.execute(sql, params)

# Print the number of rows affected
print("Rows deleted:", rp.rowcount)

Rows deleted: 4


In [11]:
sql = "SELECT name, id FROM tickers"
tickers = pd.read_sql(sql, conlt)
tickers.sample(5)

,name,id
246,ROJNA,404
7,AH,9
358,TSTH,578
291,SPC,463
326,THCOM,523


In [12]:
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
df_ins.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
0,3BBIF,2026,Q1,"6,831,513","10,827,771","-3,996,258",-36.91%,234
1,ADVANC,2026,Q1,"50,797,881","47,885,902","2,911,979",6.08%,6
2,AIT,2026,Q1,"581,809","581,113",696,0.12%,11
3,AOT,2026,Q1,"17,433,533","18,125,205","-691,672",-3.82%,24
4,ASIAN,2026,Q1,"604,361","681,832","-77,471",-11.36%,36


In [13]:
# Convert DataFrame to list of records
rcds = df_ins.values.tolist()

# Define column names in the same order as values
columns = ['name', 'year', 'quarter', 'latest_amt', 'previous_amt', 'inc_amt', 'inc_pct', 'ticker_id']

# SQL insert statement with named parameters
sql = text("""
    INSERT INTO qt_profits 
    (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id)
    VALUES (:name, :year, :quarter, :latest_amt, :previous_amt, :inc_amt, :inc_pct, :ticker_id)
""")

try:
    # Execute inserts
    for rcd in rcds:
        # Convert list to dictionary
        params = dict(zip(columns, rcd))
        conlt.execute(sql, params)
except Exception as e:
    raise e

### End of loop

In [15]:
criteria_1 = df_ins.q_amt_c > 440_000
df_ins.loc[criteria_1, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q1,"6,831,513","10,827,771","-3,996,258",-36.91%
1,ADVANC,2026,Q1,"50,797,881","47,885,902","2,911,979",6.08%
2,AIT,2026,Q1,"581,809","581,113",696,0.12%
3,AOT,2026,Q1,"17,433,533","18,125,205","-691,672",-3.82%
4,ASIAN,2026,Q1,"604,361","681,832","-77,471",-11.36%


In [16]:
criteria_2 = df_ins.q_amt_p > 400_000
df_ins.loc[criteria_2, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q1,"6,831,513","10,827,771","-3,996,258",-36.91%
1,ADVANC,2026,Q1,"50,797,881","47,885,902","2,911,979",6.08%
2,AIT,2026,Q1,"581,809","581,113",696,0.12%
3,AOT,2026,Q1,"17,433,533","18,125,205","-691,672",-3.82%
4,ASIAN,2026,Q1,"604,361","681,832","-77,471",-11.36%


In [17]:
criteria_3 = df_ins.percent > 10.00
df_ins.loc[criteria_3, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
5,ASK,2026,Q1,"587,609","531,545","56,064",10.55%
8,BCPG,2026,Q1,"1,425,162","855,451","569,711",66.60%
13,DELTA,2026,Q1,"28,407,374","24,814,324","3,593,050",14.48%
20,KKP,2026,Q1,"6,806,788","5,912,913","893,875",15.12%
26,MST,2026,Q1,"264,547","220,510","44,037",19.97%


In [18]:
df_ins_criteria = criteria_1 & criteria_2 & criteria_3
df_ins.loc[df_ins_criteria, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
5,ASK,2026,Q1,"587,609","531,545","56,064",10.55%
8,BCPG,2026,Q1,"1,425,162","855,451","569,711",66.60%
13,DELTA,2026,Q1,"28,407,374","24,814,324","3,593,050",14.48%
20,KKP,2026,Q1,"6,806,788","5,912,913","893,875",15.12%
30,PSL,2026,Q1,"662,267","413,846","248,421",60.03%


In [19]:
df_ins[df_ins_criteria].sort_values(by=["percent"], ascending=[False]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
8,BCPG,2026,Q1,"1,425,162","855,451","569,711",66.60%,53
30,PSL,2026,Q1,"662,267","413,846","248,421",60.03%,734
48,TRUE,2026,Q1,"14,195,303","9,240,463","4,954,840",53.62%,571
44,TIDLOR,2026,Q1,"5,248,767","3,622,141","1,626,626",44.91%,751
35,SCC,2026,Q1,"19,199,135","14,075,020","5,124,115",36.41%,427


In [20]:
df_ins[df_ins_criteria].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
5,ASK,2026,Q1,"587,609","531,545","56,064",10.55%,38
8,BCPG,2026,Q1,"1,425,162","855,451","569,711",66.60%,53
13,DELTA,2026,Q1,"28,407,374","24,814,324","3,593,050",14.48%,138
20,KKP,2026,Q1,"6,806,788","5,912,913","893,875",15.12%,255
30,PSL,2026,Q1,"662,267","413,846","248,421",60.03%,734


In [21]:
df_ins[df_ins_criteria].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
5,ASK,2026,Q1,"587,609","531,545","56,064",10.55%,38
8,BCPG,2026,Q1,"1,425,162","855,451","569,711",66.60%,53
13,DELTA,2026,Q1,"28,407,374","24,814,324","3,593,050",14.48%,138
20,KKP,2026,Q1,"6,806,788","5,912,913","893,875",15.12%,255
30,PSL,2026,Q1,"662,267","413,846","248,421",60.03%,734


In [22]:
conlt.commit()
conlt.close()

In [23]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:05:11 22:23:44
